# Step 1: Data Cleaning – Steam Catalog Dashboard
**Mini Project 2 – Mid Submission** · Source: Crawlora (2026), *Steam Catalog Dataset – State of Steam 2026*, `data/sample.jsonl` (CC BY 4.0)

This notebook turns the raw 200-game sample into the tables the dashboard needs. Every decision is explained in the markdown cell above the code that carries it out, and each step prints the counts it affects, so the numbers in the Word cleaning notes can be traced back here.

**Pipeline position (Week 1, reference model):** Raw Data → *Data Transformations (this notebook)* → Data Tables → Visual Mappings → Views.

**Research questions this data has to serve**
- **Q1** – Is release year related to the price of paid games? (line plot)
- **Q2** – Do higher-priced paid games have more peak concurrent players? (scatterplot)
- **Q3** – For the 8 most common genres, how do review-tier shares and prices compare? (waffle chart + bar chart)

## 0. Setup
Set `SRC` to the location of `sample.jsonl` inside the unzipped dataset and `OUT` to the folder where the clean tables should be written.

In [1]:
import json
from collections import Counter

import numpy as np
import pandas as pd

SRC = "steam-catalog-data-main/data/sample.jsonl"   # path to the raw sample
OUT = "data"                                         # output folder for clean tables

import os
os.makedirs(OUT, exist_ok=True)
log = {}   # collects the counts reported in the cleaning notes

## 1. Load the raw data and check integrity
The sample is a top-200 list (by SteamSpy owner estimate), 39 columns. Before changing anything, check that it is what the README says: no duplicate games, and only `type == "game"` (no DLC or software).

In [2]:
raw = pd.read_json(SRC, lines=True)
log["raw_rows"], log["raw_cols"] = raw.shape

log["dup_appid"]     = int(raw.appid.duplicated().sum())
log["non_game_type"] = int((raw.type != "game").sum())
log["on_sale_at_crawl"] = int((raw.discount_pct > 0).sum())

print("shape:", raw.shape)
print("duplicate appids:", log["dup_appid"])
print("rows that are not type=game:", log["non_game_type"])
print("games on sale when crawled:", log["on_sale_at_crawl"])

shape: (200, 39)
duplicate appids: 0
rows that are not type=game: 0
games on sale when crawled: 41


## 2. Keep only the columns the proposal uses
The proposal's channel justification table uses 6 variables plus an identifier. I also keep `is_free`, `discount_pct`, `price_cents` (needed to check data problems below) and `owners_midpoint` (needed to rank genres, step 8). Everything else is dropped so the dashboard cannot accidentally depend on unused fields.

In [3]:
df = raw[["appid", "name", "is_free", "price_cents", "initial_price_cents", "discount_pct",
          "ccu", "review_tier", "release_year", "genres", "owners_midpoint"]].copy()
print(df.shape)
df.head(3)

(200, 11)


,appid,name,is_free,price_cents,initial_price_cents,discount_pct,ccu,review_tier,release_year,genres,owners_midpoint
0,730,Counter-Strike 2,True,0,0,0,1013936,very_positive,2012.0,"[Action, Free To Play]",150000000
1,578080,PUBG: BATTLEGROUNDS,True,0,0,0,314682,mixed,2017.0,"[Action, Adventure, Massively Multiplayer, Fre...",150000000
2,1172470,Apex Legends™,True,0,0,0,124262,mixed,2020.0,"[Action, Adventure, Free To Play]",150000000


## 3. Price: use the list price, not the sale price
**Problem.** `price_cents` is the price *at the moment of the crawl*. 41 of the 200 games were discounted that day (up to 95% off). If Q1–Q3 used `price_cents`, the "price" of a game would depend on whether it happened to be on sale, not on the game.

**Decision.** Use `initial_price_cents` (the list price) and convert it to dollars as `price_usd`.

**Course link.** Week 1 *Not neutral*: a chart can mislead at exactly one arrow of the pipeline. Here it would be a botched data transformation, so it is fixed before any encoding is chosen.

In [4]:
df["price_usd"] = df.initial_price_cents / 100

paid_now = df[~df.is_free]
log["median_paid_sale_price"] = float(paid_now.price_cents.median() / 100)
log["median_paid_list_price"] = float(paid_now.price_usd.median())
print("median paid price, sale price used :", log["median_paid_sale_price"])
print("median paid price, list price used :", log["median_paid_list_price"])

median paid price, sale price used : 19.99
median paid price, list price used : 24.99


## 4. Define "paid" and handle inconsistent free flags
**Problem.** All three questions are about *paid* games, so the 58 free games must go. But `is_free` is `True` for two games that also have a price: *Tom Clancy's Rainbow Six Siege* ($19.99) and *Life is Strange 2* ($31.96). Their price is ambiguous (a free base game with paid editions, or a free first episode).

**Decision.** Trust the `is_free` flag. Those two games are treated as free and removed with the rest. This keeps the paid set conservative: every game left has one clear list price.

In [5]:
log["free_flag_with_price"] = df[df.is_free & (df.initial_price_cents > 0)].name.tolist()
print("flagged free but carrying a price:", log["free_flag_with_price"])

log["free_rows_removed"] = int(df.is_free.sum())
paid = df[~df.is_free].copy()
log["paid_rows"] = len(paid)

assert (paid.initial_price_cents > 0).all()      # every paid game has a price above zero
print("free removed:", log["free_rows_removed"], "| paid kept:", log["paid_rows"])

flagged free but carrying a price: ["Tom Clancy's Rainbow Six Siege", 'Life is Strange 2']
free removed: 58 | paid kept: 142


## 5. Missing values
Only two paid games have a missing value in a column the dashboard uses.

| Game | Missing | Decision |
|---|---|---|
| Assassin's Creed IV Black Flag | `release_year` | Drop from **Q1 only** (needs year). Keep for Q2 and Q3. |
| Middle-earth: Shadow of Mordor | `genres` | Drop from **Q3 only** (needs genre). Keep for Q1 and Q2. |

I do **not** fill these from outside knowledge: the dataset should be reproducible from the file alone, and one fact typed in by hand is a silent edit the grader cannot see. Removing a row only from the view that cannot use it loses the least data.

In [6]:
miss_year  = paid[paid.release_year.isna()].name.tolist()
miss_genre = paid[paid.genres.isna()].name.tolist()
log["paid_missing_year"], log["paid_missing_genre"] = miss_year, miss_genre
print("missing release_year:", miss_year)
print("missing genres      :", miss_genre)

# release_year arrives as float because of the NaN; make it a nullable integer
paid["release_year"] = paid.release_year.astype("Int64")

missing release_year: ['Assassin’s Creed® IV Black Flag™']
missing genres      : ['Middle-earth™: Shadow of Mordor™']


## 6. Types: make review tier an ordered (ordinal) variable
**Problem.** `review_tier` is text (`"very_positive"`), and text sorts alphabetically, which would put *Mixed* between *Mostly Positive* and *Very Positive* wrongly.

**Decision.** Add a rank following Steam's own order, plus a readable label for the legend:
`Mixed (1) < Mostly Positive (2) < Very Positive (3) < Overwhelmingly Positive (4)`.
Only these four tiers occur among paid games, which matches the four-level legend in the proposal (and is under the 7-item limit from Week 6).

**Course link.** Week 1 channel ranking: an *ordered* attribute is matched to an ordered channel (luminance), not an identity channel (hue). That only works if the order is encoded in the data.

In [7]:
tier_order = ["mixed", "mostly_positive", "very_positive", "overwhelmingly_positive"]
tier_label = {"mixed": "Mixed", "mostly_positive": "Mostly Positive",
              "very_positive": "Very Positive", "overwhelmingly_positive": "Overwhelmingly Positive"}

assert set(paid.review_tier) <= set(tier_order), "unexpected review tier in paid games"

paid["review_tier_rank"]  = paid.review_tier.map({t: i + 1 for i, t in enumerate(tier_order)})
paid["review_tier_label"] = paid.review_tier.map(tier_label)
print(paid.review_tier_label.value_counts().reindex([tier_label[t] for t in tier_order]))

review_tier_label
Mixed                       8
Mostly Positive            14
Very Positive              63
Overwhelmingly Positive    57
Name: count, dtype: int64


## 7. Display names for tooltips
**Problem.** 23 paid titles contain ® and ™ (e.g. *Call of Duty®*) and some use a curly apostrophe. These render inconsistently and clutter tooltips.

**Decision.** Add `name_clean` for tooltips. The original `name` stays in the table.

**Course link.** Week 1 *Not decoration* (data-ink): symbols that carry no information add ink and subtract meaning.

In [8]:
log["names_with_symbols"] = int(paid.name.str.contains(r"[®™]").sum())
paid["name_clean"] = (paid.name.str.replace(r"[®™]", "", regex=True)
                               .str.replace("’", "'")
                               .str.strip())
print("names with ®/™:", log["names_with_symbols"])
paid.loc[paid.name != paid.name_clean, ["name", "name_clean"]].head(5)

names with ®/™: 23


,name,name_clean
7,Call of Duty®,Call of Duty
17,HELLDIVERS™ 2,HELLDIVERS 2
56,Sid Meier’s Civilization® VI,Sid Meier's Civilization VI
60,Sekiro™: Shadows Die Twice - GOTY Edition,Sekiro: Shadows Die Twice - GOTY Edition
61,Battlefield™ 2042,Battlefield 2042


## 8. Peak concurrent players (`ccu`): check it can sit on a log axis
**Finding.** `ccu` runs from 1 to 143,870 with a median of about 5,000: roughly five orders of magnitude and heavily right-skewed. On a linear axis most games would be squeezed into the bottom of the chart.

**Decision.** Keep the raw values (no transformation in the data). The log scale is applied in the chart, so axis ticks still show real player counts. A log axis needs every value above zero, so that is asserted here.

**Update to the proposal text.** The proposal's "196 of 200 games sit in the bottom 10% of a 0–1M axis; 88 are between 1k and 10k" describes **all 200 games**, including Counter-Strike 2 (about 1.0M). For the 142 **paid** games the numbers are below. The log axis is still justified; only the figures in the table should change.

In [9]:
log["ccu_min"], log["ccu_max"], log["ccu_median"] = int(paid.ccu.min()), int(paid.ccu.max()), float(paid.ccu.median())
log["ccu_1k_10k"]   = int(((paid.ccu >= 1000) & (paid.ccu <= 10000)).sum())
log["ccu_over_100k"] = int((paid.ccu > 100000).sum())

assert (paid.ccu > 0).all()          # required for a log axis
print("ccu min / median / max:", log["ccu_min"], "/", log["ccu_median"], "/", log["ccu_max"])
print("paid games with ccu between 1k and 10k:", log["ccu_1k_10k"], "of", len(paid))
print("paid games above 100k                  :", log["ccu_over_100k"])

ccu min / median / max: 1 / 5163.0 / 143870
paid games with ccu between 1k and 10k: 70 of 142
paid games above 100k                  : 1


## 9. Q3: choose the top 8 genres among *paid* games
**Rule from the proposal.** The 8 genres with the most ownership (sum of `owners_midpoint`). A game can carry several genres, so it counts once in each of its genres.

**Problem.** Ranked over all 200 games, **Free To Play** lands in the top 8. It is a pricing model, not a genre, and it has **zero paid games**, so it has no median price for the bar chart.

**Decision.** Rank using paid games only. **Strategy** takes Free To Play's place. Same rule, applied to the population the question is actually about.

In [10]:
own, cnt = Counter(), Counter()
for gs, o in zip(paid.genres, paid.owners_midpoint):
    if isinstance(gs, list):
        for g in gs:
            own[g] += o
            cnt[g] += 1

all_own = Counter()
for gs, o in zip(raw.genres, raw.owners_midpoint):
    if isinstance(gs, list):
        for g in gs:
            all_own[g] += o

top8 = [g for g, _ in own.most_common(8)]
log["genres_all_200"] = len({g for gs in raw.genres.dropna() for g in gs})
log["genres_paid"]    = len(own)
log["top8"]           = [(g, int(cnt[g]), int(own[g])) for g in top8]
log["top8_if_all_200"] = [g for g, _ in all_own.most_common(8)]

print("top 8 (paid only)   :", top8)
print("top 8 (all 200 games):", log["top8_if_all_200"])
print("paid games per genre:", {g: int(cnt[g]) for g in top8})

paid["genres_str"] = paid.genres.apply(lambda g: "|".join(g) if isinstance(g, list) else "")
paid["in_top8"]    = paid.genres.apply(lambda g: isinstance(g, list) and any(x in top8 for x in g))

top 8 (paid only)   : ['Action', 'Adventure', 'RPG', 'Indie', 'Simulation', 'Strategy', 'Massively Multiplayer', 'Casual']
top 8 (all 200 games): ['Action', 'Adventure', 'Free To Play', 'RPG', 'Indie', 'Massively Multiplayer', 'Simulation', 'Casual']
paid games per genre: {'Action': 103, 'Adventure': 62, 'RPG': 45, 'Indie': 48, 'Simulation': 30, 'Strategy': 25, 'Massively Multiplayer': 13, 'Casual': 10}


## 10. Q1: aggregate by year and choose the window
**Aggregation.** Q1 plots the **mean and median list price of paid games per release year** (as the proposal describes).

**Problem.** The sample holds only ~9 paid games per year, and far fewer before 2012: 1 to 4 per year, with several years empty. A median of one game is that game's price, not a trend. The dataset README also warns that 2025–2026 are small samples and that the robust trend runs 2012→2024.

**Decision.** Trend window = **2012–2024** with at least 5 games per year. Earlier years and 2025–2026 are *not deleted*: `price_by_year.csv` keeps every year with its `n` and an `in_trend_window` flag, so the dashboard can show `n` in tooltips or grey the excluded years out.

**Limit to state on the dashboard.** This is the top 200 by owners, so it skews to popular, big-budget games. Median list prices here ($20–$60) are much higher than the README's study-wide medians ($5–$10), and older years are survivors. Q1 describes *popular* games.

In [11]:
yr = (paid.dropna(subset=["release_year"])
          .groupby("release_year").price_usd
          .agg(n="count", median="median", mean="mean")
          .reset_index())
yr["in_trend_window"] = yr.release_year.between(2012, 2024) & (yr.n >= 5)

win = yr[yr.in_trend_window]
log["q1_rows"]  = int(win.n.sum())
log["q1_years"] = int(len(win))
log["q1_n_range"] = [int(win.n.min()), int(win.n.max())]
log["q1_n_mean"]  = round(float(win.n.mean()), 1)
log["q1_excluded_years"] = yr[~yr.in_trend_window][["release_year", "n"]].astype(int).values.tolist()
log["q1_excluded_rows"]  = int(yr[~yr.in_trend_window].n.sum())

print(f"Q1 window: {log['q1_years']} years, {log['q1_rows']} games, {log['q1_n_range'][0]}-{log['q1_n_range'][1]} per year (mean {log['q1_n_mean']})")
print("excluded (year, n):", log["q1_excluded_years"])
yr

Q1 window: 13 years, 118 games, 5-14 per year (mean 9.1)
excluded (year, n): [[1998, 1], [2000, 1], [2004, 4], [2005, 1], [2006, 1], [2007, 1], [2008, 1], [2009, 1], [2010, 3], [2011, 3], [2025, 5], [2026, 1]]


,release_year,n,median,mean,in_trend_window
0,1998,1,9.99,9.990000,False
1,2000,1,9.99,9.990000,False
2,2004,4,9.99,12.490000,False
3,2005,1,19.99,19.990000,False
4,2006,1,9.99,9.990000,False
5,2007,1,9.99,9.990000,False
6,2008,1,9.99,9.990000,False
7,2009,1,9.99,9.990000,False
8,2010,3,19.99,19.990000,False
9,2011,3,9.99,13.323333,False


## 11. Q3: genre table, review-tier shares and waffle squares
**Long table.** Because a game can have several genres, build one row per *(game, genre)* for the top-8 genres. This is the table the bar chart and waffle chart share, so a click on one genre can filter every view (linking, rubric criterion 4).

**Waffle rounding.** The proposal uses 20 squares per genre, each worth 5%. Shares are rounded to whole squares with the **largest-remainder method**, so every genre row has exactly 20 squares (no row is longer or shorter because of rounding).

**Course link.** Week 1: length/position on a common scale is the most accurate magnitude channel, so each genre gets one row of equal length. The cost, stated in the proposal, is precision: 26% and 28% both round to 5 squares.

**Small groups.** Casual has 10 paid games and Massively Multiplayer 13, so one game moves a row by 1.5–2 squares and the median price is unstable. This should be shown on the dashboard (e.g. "n = 10").

In [12]:
rows = []
for _, r in paid[paid.genres.notna()].iterrows():
    for g in r.genres:
        if g in top8:
            rows.append(dict(appid=r.appid, name_clean=r.name_clean, genre=g,
                             price_usd=r.price_usd, ccu=r.ccu, release_year=r.release_year,
                             review_tier=r.review_tier, review_tier_rank=r.review_tier_rank))
long = pd.DataFrame(rows)
log["q3_games"]           = int(long.appid.nunique())
log["q3_game_genre_rows"] = len(long)

summ = []
for g in top8:
    s = long[long.genre == g]
    n = len(s)
    shares = s.review_tier.value_counts().reindex(tier_order, fill_value=0) / n * 20
    sq  = np.floor(shares).astype(int)
    rem = shares - sq
    for t in rem.sort_values(ascending=False).index[: 20 - sq.sum()]:
        sq[t] += 1                                   # give leftover squares to the largest remainders
    d = dict(genre=g, n_paid_games=n, median_price_usd=s.price_usd.median(),
             mean_price_usd=s.price_usd.mean(), owners_sum=int(own[g]))
    for t in tier_order:
        d[f"n_{t}"] = int((s.review_tier == t).sum())
        d[f"squares_{t}"] = int(sq[t])
    assert sum(d[f"squares_{t}"] for t in tier_order) == 20
    summ.append(d)

summ = pd.DataFrame(summ).sort_values("median_price_usd", ascending=False)   # bar chart: descending order
log["genre_n_min"] = int(summ.n_paid_games.min())
print("games in Q3:", log["q3_games"], "| game-genre rows:", log["q3_game_genre_rows"])
summ[["genre", "n_paid_games", "median_price_usd"] + [f"squares_{t}" for t in tier_order]]

games in Q3: 141 | game-genre rows: 336


,genre,n_paid_games,median_price_usd,squares_mixed,squares_mostly_positive,squares_very_positive,squares_overwhelmingly_positive
5,Strategy,25,35.00,0,3,9,8
6,Massively Multiplayer,13,34.99,3,8,9,0
0,Action,103,29.99,1,2,9,8
1,Adventure,62,29.99,1,3,9,7
2,RPG,45,29.99,1,3,9,7
4,Simulation,30,27.49,2,1,9,8
3,Indie,48,19.99,0,1,8,11
7,Casual,10,12.49,2,0,4,14


## 12. Write the output tables

| File | Rows | Feeds |
|---|---|---|
| `steam_paid_clean.csv` | 142 | Q1, Q2 (one row per paid game) |
| `steam_paid_top8_genre_long.csv` | 336 | Q3 and cross-view linking (one row per game and genre) |
| `price_by_year.csv` | 25 | Q1 mean/median line, with `n` and trend-window flag |
| `genre_summary_waffle.csv` | 8 | Q3 bar chart and waffle squares |
| `cleaning_log.json` | – | counts used in the Word cleaning notes |

In [13]:
paid_out = paid[["appid", "name", "name_clean", "price_usd", "price_cents", "discount_pct", "ccu",
                 "release_year", "review_tier", "review_tier_label", "review_tier_rank",
                 "genres_str", "in_top8", "owners_midpoint"]]

paid_out.to_csv(f"{OUT}/steam_paid_clean.csv", index=False)
long.to_csv(f"{OUT}/steam_paid_top8_genre_long.csv", index=False)
yr.to_csv(f"{OUT}/price_by_year.csv", index=False)
summ.to_csv(f"{OUT}/genre_summary_waffle.csv", index=False)
with open(f"{OUT}/cleaning_log.json", "w") as f:
    json.dump(log, f, indent=1, default=str)

for name, d in [("steam_paid_clean", paid_out), ("steam_paid_top8_genre_long", long),
                ("price_by_year", yr), ("genre_summary_waffle", summ)]:
    print(f"{name}.csv: {len(d)} rows x {d.shape[1]} cols")

steam_paid_clean.csv: 142 rows x 14 cols
steam_paid_top8_genre_long.csv: 336 rows x 8 cols
price_by_year.csv: 25 rows x 5 cols
genre_summary_waffle.csv: 8 rows x 13 cols


## Summary of decisions

1. **Price** = list price (`initial_price_cents`), because 41 games were on sale at crawl time.
2. **Paid** = `is_free == False`; two games flagged free but carrying a price are treated as free.
3. **Missing values** are dropped only from the one question that needs them, and are not imputed.
4. **Review tier** becomes an ordered 1–4 variable so it can use an ordered channel (luminance).
5. **ccu** keeps raw values; log scale is applied in the chart. All values are above zero.
6. **Top 8 genres** are ranked on paid games, so Strategy replaces Free To Play.
7. **Q1 window** is 2012–2024 with at least 5 games per year; other years stay in the file but are flagged.
8. **Waffle** rows are rounded to exactly 20 squares with the largest-remainder method.

**Limitations:** top-200 sample skews to popular, higher-priced games; older years are survivors; Casual (n = 10) and Massively Multiplayer (n = 13) are small groups; owners and `ccu` are estimates from one snapshot.